# 📝 InFi-Check — Summary Generation Pipeline (per-folder)

Notebook này thực hiện **Bước 3** trong pipeline InFi-Check:
```
Dataset NLP/<category>/*.txt  ->  [summary_gen]  ->  new_summary/<category>/<file>_summary.txt
```

| Bước | File | Mô tả |
|------|------|-------|
| 1 | `dataset_*.py` | Làm sạch data |
| 2 | ✅ (đã xong) | Upload document lên Drive |
| **3** | **`summary_gen.ipynb`** | **← Notebook này** |
| 4 | `eval_and_reference_gen.py` | Refine + tìm câu hỗ trợ |
| 5 | `structured_dataset_gen.py` | Sinh error data |
| 6 | `prepare_dataset_base.py` | Xuất JSONL |

### Chiến lược lọc đầu vào theo quy mô folder
| Số bài trong folder | Chiến lược |
|---|---|
| **> 250 bài** | Lọc chất lượng cao: từ số (150-1200), type-token ratio, mật độ thực thể, dedup Jaccard bigram — giữ tối đa 250 bài tốt nhất |
| **<= 250 bài** | Chạy thẳng toàn bộ (chỉ lọc range 150-1500 từ) |


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Cấu hình đường dẫn

In [ ]:
import os

# ✏️ CHỈNH các đường dẫn phù hợp với Drive của bạn
BASE_DOC_FOLDER   = '/content/drive/MyDrive/Dataset NLP'
PROJECT_ROOT      = '/content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/selected_dataset'
BASE_SUMMARY_ROOT = os.path.join(PROJECT_ROOT, 'new_summary')

os.makedirs(BASE_SUMMARY_ROOT, exist_ok=True)

# Hiển thị danh sách folder sẽ xử lý
categories = sorted([
    d for d in os.listdir(BASE_DOC_FOLDER)
    if os.path.isdir(os.path.join(BASE_DOC_FOLDER, d))
])
print(f'📂 Doc root    : {BASE_DOC_FOLDER}')
print(f'📂 Summary root: {BASE_SUMMARY_ROOT}')
print(f'\n📋 Danh sách {len(categories)} folder:')
for cat in categories:
    n = len([f for f in os.listdir(os.path.join(BASE_DOC_FOLDER, cat)) if f.endswith('.txt')])
    tag = '  ⚡ se loc' if n > 250 else ''
    print(f'   {cat:<35} {n:>5} bai{tag}')


📂 Doc root    : /content/drive/MyDrive/Dataset NLP
📂 Summary root: /content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/selected_dataset/new_summary

📋 Danh sách 27 folder:
   Bat_dong_san                          291 bai  ⚡ se loc
   Chinh_tri                             777 bai  ⚡ se loc
   Chong_Dien_Bien_Hoa_Binh               20 bai
   Cong_doan                              40 bai
   Cong_nghe                             633 bai  ⚡ se loc
   Doi_song                              851 bai  ⚡ se loc
   Du_lich                               356 bai  ⚡ se loc
   Giai_tri                             1063 bai  ⚡ se loc
   Giao_duc                              891 bai  ⚡ se loc
   Khoa_hoc                              310 bai  ⚡ se loc
   Kinh_doanh                            448 bai  ⚡ se loc
   Kinh_te                               596 bai  ⚡ se loc
   Moi_truong                             62 bai
   Oto_xe_may                            250 bai
   Phap_luat              

## 3. Cài đặt thư viện

In [ ]:
!pip install -q openai


## 4. Cấu hình API key & model

> 🔑 **Khuyên dùng Colab Secrets:** `Tools -> Secrets -> Add new secret -> GROQ_API_KEY`


In [ ]:
from google.colab import userdata

# Sử dụng DEEPSEEK_API_KEY từ Secrets
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

# Cấu hình cho DeepSeek
MODEL_NAME = 'deepseek-chat'
API_BASE   = 'https://api.deepseek.com'

print(f'Model  : {MODEL_NAME}')
print(f'API key: {DEEPSEEK_API_KEY[:8]}...{DEEPSEEK_API_KEY[-4:]}' if DEEPSEEK_API_KEY else '❌ Chưa có DEEPSEEK_API_KEY!')

Model  : deepseek-chat
API key: sk-693a1...01b3


## 5. Định nghĩa SummaryGenerator

In [ ]:
import re
import time
import math
from openai import OpenAI, BadRequestError
from requests.exceptions import RequestException

WORD_MIN = 150
WORD_MAX = 700


# ── Hàm lọc chất lượng cho folder lớn ──────────────────────────────────────
def _ttr(text):
    """Type-Token Ratio: tỉ lệ từ duy nhất / tổng từ (đo độ phong phú từ vựng)"""
    words = text.lower().split()
    return len(set(words)) / len(words) if words else 0


def _entity_density(text):
    """Mật độ thực thể đơn giản: đếm token bắt đầu bằng chữ HOA / tổng từ"""
    words = text.split()
    if not words:
        return 0
    named = sum(1 for w in words if w and w[0].isupper() and not w.isupper())
    return named / len(words)


def _bigrams(text):
    """Tập bigram từ của văn bản"""
    words = text.lower().split()
    return set(zip(words, words[1:]))


def _jaccard(set_a, set_b):
    if not set_a or not set_b:
        return 0.0
    inter = len(set_a & set_b)
    union = len(set_a | set_b)
    return inter / union if union else 0.0


def _quality_filter(file_paths, limit, word_min=150, word_max=1200, jaccard_thresh=0.5):
    """
    Lọc và chọn `limit` bài tốt nhất từ danh sách file:
      1. Lọc range từ [word_min, word_max]
      2. Tính score = TTR * 0.5 + entity_density * 0.5
      3. Sắp xếp giảm dần theo score
      4. Dedup Jaccard bigram: bỏ bài quá giống bài đã chọn (>= jaccard_thresh)
      5. Lấy tối đa `limit` bài
    """
    scored = []
    for path in file_paths:
        try:
            text = open(path, encoding='utf-8').read()
        except Exception:
            continue
        words = text.split()
        n = len(words)
        if not (word_min <= n <= word_max):
            continue
        score = _ttr(text) * 0.5 + _entity_density(text) * 0.5
        scored.append((score, path, text))

    scored.sort(key=lambda x: x[0], reverse=True)

    selected = []
    selected_bigrams = []
    for score, path, text in scored:
        bg = _bigrams(text)
        # Kiểm tra trùng lặp với tất cả bài đã chọn
        if any(_jaccard(bg, sb) >= jaccard_thresh for sb in selected_bigrams):
            continue
        selected.append(path)
        selected_bigrams.append(bg)
        if len(selected) >= limit:
            break

    return selected


class SummaryGenerator:
    """
    - Folder > LIMIT_PER_FOLDER: lọc chất lượng cao (TTR, entity density, dedup Jaccard bigram)
    - Folder <= LIMIT_PER_FOLDER: lấy hết (lọc range 150-1500 từ)
    - Resume-safe, retry 5 lần, temperature=0.3
    """

    def __init__(self, api_key, model_name, base_url):
        self.client     = OpenAI(api_key=api_key, base_url=base_url)
        self.model_name = model_name

    def _build_messages(self, document, doc_words):
        words = document.split()
        if len(words) > 1200:
            document  = ' '.join(words[:1200])
            doc_words = 1200
        max_words = int(min(150, max(60, doc_words / 8)))
        return [
            {'role': 'system', 'content': 'Ban la mot tro ly huu ich, thanh thao tieng Viet.'},
            {'role': 'user', 'content': (
                f'Toi se cung cap cho ban mot bai bao tieng Viet. '
                f'Nhiem vu cua ban la viet tom tat ngan cho bai bao nay theo cac yeu cau sau:\n'
                f'1. Do dai tom tat trong khoang 60 den {max_words} tu.\n'
                f'2. Moi cau trong tom tat phai duoc ho tro truc tiep boi noi dung tai lieu.\n'
                f'3. Doi voi moi su kien, hay giu lai cac thuc the quan trong nhu nguoi, dia diem '
                f'va thoi gian, dac biet la cac thuc the xuat hien song song.\n'
                f'4. Khi don gian hoa, dam bao moi su kien hoac y tuong phuc tap van trung thuc '
                f'voi y nghia goc. Tranh don gian hoa qua muc dan den mau thuan voi tai lieu goc.\n'
                f'5. Viet tom tat bang tieng Viet.\n'
                f'6. Giu nguyen ten rieng, so lieu, ngay thang.\n'
                f'7. Khong dung dai tu thay the (ong, ba, ho, anh...) neu chua de cap doi tuong trong cung cau do.\n\n'
                f'Tai lieu:\n{document}\n\n'
                f'Hay xuat truc tiep tom tat ma khong co bat ky tu thua nao.'
            )}
        ]

    def _call_api(self, messages, doc_words):
        min_words = 30
        max_words = int(min(150, max(60, doc_words / 8)))
        for attempt in range(1, 6):
            try:
                resp    = self.client.chat.completions.create(
                    model=self.model_name, messages=messages, temperature=0.3
                )
                answer  = resp.choices[0].message.content
                if 'deepseek-r1' in self.model_name.lower():
                    answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL)
                answer  = answer.strip()
                n_words = len(answer.split())
                print(f'  [attempt {attempt}] {resp.usage.prompt_tokens}p/{resp.usage.completion_tokens}c tok '
                      f'| summary: {n_words} tu (min {min_words})')
                if n_words >= min_words:
                    return answer
                print('  ⚠️  Qua ngan, thu lai...')
            except RequestException as e:
                wait = 2 ** attempt
                print(f'  ❌ Loi mang attempt {attempt}: {e} - cho {wait}s')
                time.sleep(wait)
            except BadRequestError as e:
                print(f'  ❌ BadRequest: {e}')
                return None
            except Exception as e:
                print(f'  ❌ Loi khac attempt {attempt}: {e}')
        return None

    def process_category(self, category, input_folder, output_folder, limit):
        cat_out = os.path.join(output_folder, category)
        os.makedirs(cat_out, exist_ok=True)

        all_files = sorted([
            os.path.join(input_folder, f)
            for f in os.listdir(input_folder) if f.endswith('.txt')
        ])
        total_raw = len(all_files)

        print(f'\n{"="*60}')
        if total_raw > limit:
            print(f'📂 [{category}]  {total_raw} bai > {limit}  ->  loc chat luong, giu {limit} bai tot nhat')
            candidates = _quality_filter(all_files, limit=limit, word_min=150, word_max=1200)
            print(f'   -> {len(candidates)} bai sau loc chat luong + dedup Jaccard bigram')
        else:
            print(f'📂 [{category}]  {total_raw} bai <= {limit}  ->  lay het')
            candidates = [
                p for p in all_files
                if WORD_MIN <= len(open(p, encoding='utf-8').read().split()) <= WORD_MAX
            ]
            print(f'   -> {len(candidates)} bai vao hang doi')

        stats = {'done': 0, 'skipped_exists': 0, 'failed': 0}

        for doc_path in candidates:
            fname    = os.path.basename(doc_path)
            out_path = os.path.join(cat_out, f'{fname[:-4]}_summary.txt')

            if os.path.exists(out_path):
                stats['skipped_exists'] += 1
                continue

            with open(doc_path, 'r', encoding='utf-8') as f:
                document = f.read()
            doc_words = len(document.split())
            print(f'\n  --- {fname} ({doc_words} tu) ---')

            summary = self._call_api(self._build_messages(document, doc_words), doc_words)

            if summary is None:
                print('  ❌ Bo qua')
                stats['failed'] += 1
                continue

            with open(out_path, 'w', encoding='utf-8') as f:
                f.write(summary)
            print(f'  ✅ Luu: {out_path}')
            stats['done'] += 1

        print(f'\n  📊 [{category}] ✅:{stats["done"]} | ⏭️:{stats["skipped_exists"]} | ❌:{stats["failed"]}')
        return stats


print('✅ SummaryGenerator san sang')


✅ SummaryGenerator san sang


## 6. Chạy từng folder (per-folder)

- `CATEGORIES_TO_RUN = None` → chạy tất cả folder
- `CATEGORIES_TO_RUN = ['Kinh_te', 'Y_te']` → chỉ chạy folder chỉ định
- `LIMIT_PER_FOLDER = 50` → giới hạn 50 bài mỗi folder mỗi lần chạy


In [ ]:
# ✏️ CẤU HÌNH
CATEGORIES_TO_RUN = [
    'Van_hoa',
    'Thi_truong',
    'Moi_truong',
    'So_hoa',
    'Tam_su',
    'Y_kien',
    'Cong_doan',
    'Chong_Dien_Bien_Hoa_Binh','Khoa_hoc','Du_lich']
LIMIT_PER_FOLDER  = 200

# Khởi tạo generator với DEEPSEEK_API_KEY
generator = SummaryGenerator(DEEPSEEK_API_KEY, MODEL_NAME, API_BASE)

run_list = (
    sorted([d for d in os.listdir(BASE_DOC_FOLDER)
            if os.path.isdir(os.path.join(BASE_DOC_FOLDER, d))])
    if CATEGORIES_TO_RUN is None else CATEGORIES_TO_RUN
)
print(f'Se xu ly {len(run_list)} folder | LIMIT_PER_FOLDER = {LIMIT_PER_FOLDER}')

all_stats = {}
for category in run_list:
    cat_input = os.path.join(BASE_DOC_FOLDER, category)
    if not os.path.isdir(cat_input):
        print(f'⚠️  Khong tim thay: {cat_input}')
        continue
    all_stats[category] = generator.process_category(
        category     = category,
        input_folder = cat_input,
        output_folder= BASE_SUMMARY_ROOT,
        limit        = LIMIT_PER_FOLDER,
    )

print(f'\n{"="*55}')
print(f'{"Category":<30} {"Moi":>6} {"San":>6} {"Loi":>6}')
print('-'*48)
td = te = tf = 0
for cat, s in all_stats.items():
    print(f'{cat:<30} {s["done"]:>6} {s["skipped_exists"]:>6} {s["failed"]:>6}')
    td += s['done']; te += s['skipped_exists']; tf += s['failed']
print('-'*48)
print(f'{"TONG":<30} {td:>6} {te:>6} {tf:>6}')

Se xu ly 10 folder | LIMIT_PER_FOLDER = 200

📂 [Van_hoa]  200 bai <= 200  ->  lay het
   -> 41 bai vao hang doi

  📊 [Van_hoa] ✅:0 | ⏭️:41 | ❌:0

📂 [Thi_truong]  80 bai <= 200  ->  lay het
   -> 10 bai vao hang doi

  📊 [Thi_truong] ✅:0 | ⏭️:10 | ❌:0

📂 [Moi_truong]  62 bai <= 200  ->  lay het
   -> 2 bai vao hang doi

  📊 [Moi_truong] ✅:0 | ⏭️:2 | ❌:0

📂 [So_hoa]  60 bai <= 200  ->  lay het
   -> 27 bai vao hang doi

  📊 [So_hoa] ✅:0 | ⏭️:27 | ❌:0

📂 [Tam_su]  60 bai <= 200  ->  lay het
   -> 34 bai vao hang doi

  --- Vợ_đẹp_con_khôn,_chồng_vẫn_tìm_đến_nữ_đồng_nghiệp.txt (636 tu) ---
  [attempt 1] 1605p/140c tok | summary: 67 tu (min 30)
  ✅ Luu: /content/drive/MyDrive/Phosphor-Bai-InFi-Check/InFi-Check construct/selected_dataset/new_summary/Tam_su/Vợ_đẹp_con_khôn,_chồng_vẫn_tìm_đến_nữ_đồng_nghiệp_summary.txt

  --- Yêu_chàng_trai_kém_nhiều_tuổi,_liệu_tôi_có_hối_hận.txt (300 tu) ---
  [attempt 1] 899p/107c tok | summary: 57 tu (min 30)

## 8. Xem trước kết quả

In [ ]:
import random

print(f'{"Category":<30} {"Summaries":>10}')
print('-' * 45)
grand_total = 0
for cat in sorted(os.listdir(BASE_SUMMARY_ROOT)):
    cat_path = os.path.join(BASE_SUMMARY_ROOT, cat)
    if not os.path.isdir(cat_path):
        continue
    n = len([f for f in os.listdir(cat_path) if f.endswith('.txt')])
    grand_total += n
    print(f'{cat:<30} {n:>10}')
print('-' * 45)
print(f'{"TONG":<30} {grand_total:>10}')

print('\n-- Mau ngau nhien --')
all_done = [
    os.path.join(BASE_SUMMARY_ROOT, cat, f)
    for cat in os.listdir(BASE_SUMMARY_ROOT)
    if os.path.isdir(os.path.join(BASE_SUMMARY_ROOT, cat))
    for f in os.listdir(os.path.join(BASE_SUMMARY_ROOT, cat))
    if f.endswith('.txt')
]
for fpath in random.sample(all_done, min(3, len(all_done))):
    with open(fpath, 'r', encoding='utf-8') as f:
        content = f.read()
    rel = os.path.relpath(fpath, BASE_SUMMARY_ROOT)
    print(f'\n📄 {rel}')
    print(f'   ({len(content.split())} tu) {content[:250]}...')


Category                        Summaries
---------------------------------------------
Chong_Dien_Bien_Hoa_Binh                3
Cong_doan                               3
Du_lich                               200
Khoa_hoc                              200
Moi_truong                             10
So_hoa                                 50
Tam_su                                 42
Thi_truong                             18
Van_hoa                               116
Y_kien                                 43
---------------------------------------------
TONG                                  685

-- Mau ngau nhien --

📄 Thi_truong/Giá_heo_hơi_hôm_nay_5.4_Ổn_định,_cao_nhất_vẫn_68.000_đồngkg_summary.txt
   (68 tu) Giá heo hơi ngày 5.4 ổn định trên cả ba miền, dao động từ 63.000 - 68.000 đồng/kg. Miền Bắc có giá từ 63.000 - 65.000 đồng/kg, với Lai Châu 63.000 đồng/kg. Miền Trung dao động 63.000 - 66.000 đồng/kg, trong đó Hà Tĩnh 63.000 đồng/kg. Miền Nam có giá ...

📄 Khoa_hoc/Nhà_to

## ✅ Hoàn thành

Output folder structure:
```
new_summary/
  Bat_dong_san/   bai1_summary.txt ...
  Cong_nghe/      bai1_summary.txt ...
  ...
```

Bước tiếp theo: `eval_and_reference_gen.py`
